# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haroonrana330/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Feature leakage audit
This notebook formally verifies the leakage precautions already applied in the Week-5 model (w05_model.ipynb): confirming label-derived columns are excluded from features, demonstrating via injection test that including them would inflate the score, checking the label's base rate, and confirming no product-decision-derived flags were used as inputs.

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

if not os.path.exists("/content/flyrank-ml"):
    !git clone https://github.com/haroonrana330/flyrank-ml.git /content/flyrank-ml
os.chdir("/content/flyrank-ml")

url = "https://raw.githubusercontent.com/haroonrana330/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

print("=" * 70)
print("LEAKAGE CHECK 1: label-derived columns excluded from features?")
print("=" * 70)
excluded = ["trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"]
feature_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
                 "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
print("Excluded (label-derived or ID columns):", excluded)
print("Used as features:", feature_cols)
print("Overlap (should be EMPTY):", set(excluded) & set(feature_cols))

print()
print("=" * 70)
print("LEAKAGE CHECK 2: does adding the suspect column inflate the score?")
print("=" * 70)

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(df[col].median())

X = df[feature_cols]
y = df["is_declining_label"]
groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Baseline: clean features only
model_clean = LogisticRegression(max_iter=1000)
model_clean.fit(X_train, y_train)
auc_clean = roc_auc_score(y_test, model_clean.predict_proba(X_test)[:, 1])

# Deliberately inject a leaky feature (trend_pct, which the label is derived from)
trend_pct_clean = pd.to_numeric(df["trend_pct"], errors="coerce")
trend_pct_clean = trend_pct_clean.fillna(trend_pct_clean.median())

X_leaky_train = X_train.copy()
X_leaky_test = X_test.copy()
X_leaky_train["trend_pct_LEAKY"] = trend_pct_clean.loc[X_train.index]
X_leaky_test["trend_pct_LEAKY"] = trend_pct_clean.loc[X_test.index]

model_leaky = LogisticRegression(max_iter=1000)
model_leaky.fit(X_leaky_train, y_train)
auc_leaky = roc_auc_score(y_test, model_leaky.predict_proba(X_leaky_test)[:, 1])

print(f"AUC with clean features only:        {auc_clean:.4f}")
print(f"AUC with trend_pct injected (leaky): {auc_leaky:.4f}")
print()
if auc_leaky > auc_clean + 0.15:
    print("VERDICT: Test harness confirmed working -- injecting the leaky column")
    print("causes a large score jump, as expected. This confirms our exclusion")
    print("of trend_direction/trend_pct from the real model (w05) was correct")
    print("and necessary; without excluding it, the model would be 'cheating'.")
else:
    print("VERDICT: Unexpected -- leaky injection did not inflate the score as")
    print("expected. Test harness itself may need review.")

print()
print("=" * 70)
print("LEAKAGE CHECK 3: base rate (does the label balance make sense?)")
print("=" * 70)
base_rate = y.mean()
print(f"Base rate of declining pages: {base_rate:.4f} ({base_rate*100:.1f}%)")
print("Any model's accuracy must be compared against this, not against 0.")

print()
print("=" * 70)
print("LEAKAGE CHECK 4: product-decision-derived flags used as features?")
print("=" * 70)
print("No product flags, existing scores, or system-generated decision fields")
print("(e.g. baseline_refresh_score, suggested_action_baseline) were used as")
print("model inputs anywhere in this project -- confirmed by feature_cols list above.")

LEAKAGE CHECK 1: label-derived columns excluded from features?
Excluded (label-derived or ID columns): ['trend_direction', 'trend_pct', 'is_declining_label', 'content_id', 'client_id']
Used as features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Overlap (should be EMPTY): set()

LEAKAGE CHECK 2: does adding the suspect column inflate the score?


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


AUC with clean features only:        0.5311
AUC with trend_pct injected (leaky): 0.9514

VERDICT: Test harness confirmed working -- injecting the leaky column
causes a large score jump, as expected. This confirms our exclusion
of trend_direction/trend_pct from the real model (w05) was correct
and necessary; without excluding it, the model would be 'cheating'.

LEAKAGE CHECK 3: base rate (does the label balance make sense?)
Base rate of declining pages: 0.5421 (54.2%)
Any model's accuracy must be compared against this, not against 0.

LEAKAGE CHECK 4: product-decision-derived flags used as features?
No product flags, existing scores, or system-generated decision fields
(e.g. baseline_refresh_score, suggested_action_baseline) were used as
model inputs anywhere in this project -- confirmed by feature_cols list above.


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.